# Blood Infection (Sepsis) - Data Analysis & Exploration

Comprehensive exploratory data analysis and model experimentation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    accuracy_score, f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/sepsis_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data info
print("Data Info:")
print(df.info())
print(f"\nData Types:\n{df.dtypes}")

## 2. Descriptive Statistics

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check class distribution
sepsis_counts = df['sepsis'].value_counts()
print(f"Sepsis Distribution:")
print(f"  Negative (0): {sepsis_counts[0]} ({sepsis_counts[0]/len(df)*100:.1f}%)")
print(f"  Positive (1): {sepsis_counts[1]} ({sepsis_counts[1]/len(df)*100:.1f}%)")

# Visualize
plt.figure(figsize=(8, 4))
sepsis_counts.plot(kind='bar', color=['green', 'red'])
plt.title('Sepsis Class Distribution')
plt.xlabel('Class (0=No Sepsis, 1=Sepsis)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 3. Exploratory Data Analysis

In [ ]:
# Distribution of features
features = df.columns[:-1]  # All except target

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, feature in enumerate(features):
    axes[idx].hist(df[df['sepsis']==0][feature], alpha=0.6, label='No Sepsis', bins=20, color='blue')
    axes[idx].hist(df[df['sepsis']==1][feature], alpha=0.6, label='Sepsis', bins=20, color='red')
    axes[idx].set_title(f'Distribution of {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
correlation = df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Correlation with target
print("\nCorrelation with Sepsis Target:")
print(correlation['sepsis'].sort_values(ascending=False))

## 5. Missing Values Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
print(f"Missing values:\n{missing}")
print(f"\nTotal missing: {missing.sum()}")

## 6. Model Training & Comparison

In [ ]:
# Prepare data
X = df.drop('sepsis', axis=1)
y = df['sepsis']

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Metrics
print("Random Forest Results:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Metrics
print("Logistic Regression Results:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

## 7. Model Comparison

In [ ]:
# Confusion matrices
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Random Forest - Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Logistic Regression - Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)

auc_rf = auc(fpr_rf, tpr_rf)
auc_lr = auc(fpr_lr, tpr_lr)

plt.figure(figsize=(10, 6))
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.3f})', linewidth=2)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Feature Importance

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance Score')
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.show()

print("Feature Importance Ranking:")
print(feature_importance.to_string(index=False))

## 9. Conclusions & Recommendations

In [ ]:
print("""
ANALYSIS SUMMARY
================

1. Data Overview:
   - Total samples: {} records
   - Features: {} clinical indicators
   - Class balance: {} No Sepsis, {} Sepsis

2. Key Findings:
   - No missing values requiring imputation
   - Features show distinct patterns between classes
   - Moderate to strong correlations observed

3. Model Performance:
   - Random Forest F1 Score: {:.4f}
   - Logistic Regression F1 Score: {:.4f}
   - Recommended model: {}

4. Top Predictive Features:
{}

5. Recommendations:
   - Continue with {} for production
   - Monitor model performance regularly
   - Validate on new patient cohorts
   - Consider ensemble methods for improvement
""".format(
    len(df),
    len(X.columns),
    sepsis_counts[0],
    sepsis_counts[1],
    f1_score(y_test, y_pred_rf),
    f1_score(y_test, y_pred_lr),
    'Random Forest' if f1_score(y_test, y_pred_rf) > f1_score(y_test, y_pred_lr) else 'Logistic Regression',
    feature_importance.head(3).to_string(index=False),
    'Random Forest' if f1_score(y_test, y_pred_rf) > f1_score(y_test, y_pred_lr) else 'Logistic Regression'
))